# Классификация текстов по авторам — версия 2.0

    **Цель:** Сравнить Bag of Words и Embedding, не перепутав подписи классов.

    **Алгоритм:** Сегментация здесь — нарезка документов на окна, затем классификация. Метка и имя автора берутся из единого manifest.

    Учебная модернизация 2026-10-01. Код адаптирован с Python 3.7 на Python 3.12. Исходные алгоритмы сохранены; историческая
    версия доступна в Git на коммите `a842320`. Исправления учебных ошибок описаны
    в README. Это учебное пособие, не промышленная система и не benchmark.

    По умолчанию: CPU, короткий запуск, синтетические демонстрационные данные,
    без загрузок из сети. Реальные данные включаются отдельно: `MLLAB_DATA_SOURCE`.
    Синтетические результаты нельзя выдавать за качество на реальных данных.
    После изменения настроек перезапустите ядро и выполните ячейки сверху вниз.
    Окружение и структура данных описаны в `docs/ENVIRONMENTS.md` и `docs/DATA.md`.


In [ ]:
from pathlib import Path
import os, sys
start = Path(os.environ.get('MLLAB_ROOT', str(Path.cwd()))).resolve()
root = next((p for p in (start, *start.parents) if (p / 'ml_lab' / 'runtime.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Откройте папку репозитория или задайте MLLAB_ROOT.')
sys.path.insert(0, str(root))
from ml_lab.runtime import settings
cfg = settings('text-segmentation-keras')


## Документы и честная разбивка
Сначала разделяются документы; только потом создаются перекрывающиеся окна.


In [ ]:
from ml_lab.runtime import tensorflow_setup, author_documents, author_windows, document_split
tf = tensorflow_setup(cfg)
import numpy as np
from tensorflow.keras import layers, Sequential
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
names, records = author_documents(cfg)
documents = np.array([text for text, _ in records], dtype=object)
labels = np.array([label for _, label in records])
train_idx, val_idx, test_idx = document_split(labels, cfg.seed)
assert not (set(train_idx) & set(test_idx) or set(val_idx) & set(test_idx))
vectorizer = layers.TextVectorization(max_tokens=15000, output_mode='int', standardize='lower_and_strip_punctuation')
vectorizer.adapt(tf.constant(documents[train_idx].tolist()))
window_length, window_step = (24, 12) if cfg.quick else (1500, 200)
def make_windows(indices):
    windows, targets = [], []
    for index in indices:
        tokens = vectorizer(tf.constant([documents[index]])).numpy()[0]
        tokens = tokens[tokens != 0]
        parts = author_windows(tokens, window_length, window_step)
        windows.extend(parts); targets.extend([labels[index]] * len(parts))
    if not windows: raise ValueError('Документы короче окна: уменьшите window_length.')
    return np.stack(windows), np.array(targets)
x_train, y_train = make_windows(train_idx)
x_val, y_val = make_windows(val_idx)
x_test, y_test = make_windows(test_idx)
vocabulary_size = len(vectorizer.get_vocabulary())
print('Class mapping:', dict(enumerate(names)))
print('train/val/test windows:', x_train.shape, x_val.shape, x_test.shape)


## Bag of Words и Dense
Современный TextVectorization вместо устаревшего Tokenizer; словарь обучается только на train.


In [ ]:
def bag_of_words(windows):
    output = np.zeros((len(windows), vocabulary_size), dtype='float32')
    for row, tokens in enumerate(windows): output[row, np.unique(tokens)] = 1
    output[:, 0] = 0
    return output
bow_train, bow_val, bow_test = [bag_of_words(x) for x in [x_train, x_val, x_test]]
widths = (32, 16, 8) if cfg.quick else (1200, 600, 100)
dense = Sequential([layers.Input(shape=(vocabulary_size,)), layers.Dense(widths[0], activation='relu'),
    layers.Dense(widths[1], activation='relu'), layers.Dropout(.2), layers.BatchNormalization(),
    layers.Dense(widths[2], activation='relu'), layers.Dense(len(names), activation='softmax')])
dense.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
dense.fit(bow_train, y_train, validation_data=(bow_val, y_val), epochs=1 if cfg.quick else 30, batch_size=32, verbose=0)


## Embedding и оценка на test
Историческая идея Flatten + Dense сохранена; быстрый режим уменьшает размеры.


In [ ]:
embedded = Sequential([layers.Input(shape=(window_length,), dtype='int32'),
    layers.Embedding(vocabulary_size, 16 if cfg.quick else 100), layers.SpatialDropout1D(.2),
    layers.Flatten(), layers.BatchNormalization(), layers.Dense(32 if cfg.quick else 200, activation='relu'),
    layers.Dense(16 if cfg.quick else 50, activation='relu'), layers.Dense(len(names), activation='softmax')])
embedded.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
embedded.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=1 if cfg.quick else 20, batch_size=32, verbose=0)
for model, inputs in [(dense, bow_test), (embedded, x_test)]:
    predicted = model.predict(inputs, verbose=0).argmax(axis=1)
    print(model.name, classification_report(y_test, predicted, labels=np.arange(len(names)), target_names=names, zero_division=0))


## Результат и самостоятельная работа

    Выполненные ячейки показывают формы данных, обучение и результат алгоритма.
    Успешный короткий запуск проверяет работоспособность, а не достижение заданной точности.

    1. Измените длину и шаг окна.
2. Почему нельзя случайно разделять перекрывающиеся окна одного документа между train и test?
3. Сравните Dense и Embedding.
4. Проверьте соответствие автора, файла и номера класса.

    Перед выводом о качестве используйте отдельные train/validation/test,
    документируйте источник данных и сравнивайте эксперименты при одинаковых настройках.
